# Emotion Detection RoBERTa

**Model:** j-hartmann/emotion-english-distilroberta-base | **Size:** 83MB | **Product:** prod-du6jpd5xguk6w

A DistilRoBERTa model fine-tuned for 7-class English emotion detection (anger, disgust, fear, joy, neutral, sadness, surprise). Built on distilled RoBERTa for low-latency inference, it delivers strong performance on social media text and conversational data.

## Use Cases
- Customer feedback sentiment and emotion analysis
- Social media monitoring and brand perception tracking
- Chatbot response personalization based on user emotion
- Mental health and wellbeing app sentiment tracking

In [ ]:
import boto3
import sagemaker
from sagemaker import ModelPackage

region = boto3.Session().region_name
role = sagemaker.get_execution_role()
sm_client = boto3.client('sagemaker', region_name=region)

print(f'Region: {region}')
print(f'Role: {role}')

In [ ]:
# Replace with your actual Model Package ARN from AWS Marketplace
model_package_arn = 'arn:aws:sagemaker:REGION:ACCOUNT:model-package/MODEL_PACKAGE_NAME'

# Validate ARN before deploying
if 'REGION' in model_package_arn or 'ACCOUNT' in model_package_arn or 'MODEL_PACKAGE_NAME' in model_package_arn:
    raise ValueError(
        'model_package_arn contains placeholder values. '
        'Subscribe to the model on AWS Marketplace and replace with the actual ARN.'
    )

endpoint_name = 'emotion-detection-roberta'
instance_type = 'ml.m5.xlarge'

try:
    model = ModelPackage(
        role=role,
        model_package_arn=model_package_arn,
        sagemaker_session=sagemaker.Session()
    )
    predictor = model.deploy(
        initial_instance_count=1,
        instance_type=instance_type,
        endpoint_name=endpoint_name
    )
    print(f'Endpoint deployed: {endpoint_name}')
except Exception as e:
    print(f'Deployment failed: {e}')
    raise

## Step 2: Run Inference

Send English text to classify into one of 7 emotion categories: anger, disgust, fear, joy, neutral, sadness, or surprise.

In [ ]:
import json

runtime = boto3.client('sagemaker-runtime', region_name=region)

# Sample texts representing different emotions
texts = [
    'I just got promoted at work! This is the best day of my life!',
    'The package arrived damaged and customer service refused to help.',
    'I cannot believe the ending of that movie, I did not see that coming at all!',
]

for text in texts:
    payload = json.dumps({'inputs': text})
    try:
        response = runtime.invoke_endpoint(
            EndpointName=endpoint_name,
            ContentType='application/json',
            Body=payload
        )
        result_raw = response['Body'].read().decode('utf-8')
        try:
            result = json.loads(result_raw)
            print(f'Text: "{text[:60]}..."')
            print(f'Emotion: {result}\n')
        except json.JSONDecodeError:
            print(f'Raw response: {result_raw}')
    except Exception as e:
        print(f'Inference failed: {e}')
        raise

In [ ]:
# Cleanup - delete the endpoint to avoid ongoing charges
try:
    sm_client.delete_endpoint(EndpointName=endpoint_name)
    print(f'Endpoint {endpoint_name} deleted.')
except Exception as e:
    print(f'Cleanup failed: {e}')